# TP4.1

## TP4.1 a

Les racines du polynôme $(R_0 - 1)i - R_0 i^2$ sont $i = 0$ et $i = 1 - R_0^{-1}$ (où $R_0 \neq 0$).

$$
i^*_1 = 0
$$

$$
i^*_2 = 1 - R_0^{-1} \text{ } (\text{où } R_0 \geq 1)
$$

## TP4.1 b


À l'aide de Wolfram Alpha, on obtient (où $R_0 \neq 0$) :
$$
i(\tau) = \frac{R_0 - 1}{R_0 - \exp[(C_1 - x)(R_0 - 1)]}
$$

Où $C_1$ est une constante d'intégration. Pour $\tau = 0$, on note :
$$
i(0) = \frac{R_0 - 1}{R_0 - \exp[C_1(R_0 - 1)]}
$$

In [ ]:
import numpy as np

def i_soln(tau, R_0=1.0, i_0=0):
    #return (R_0 - 1)*np.exp(C_1 + R_0*tau)/(R_0*np.exp(C_1 + R_0*tau) - np.exp(C_1*R_0+tau))
    #return (R_0 - 1)/(R_0 - np.exp((C_1-tau)*(R_0-1)))
    K = (R_0 - 1)/i_0 - R_0

    return (R_0 - 1) / (R_0 + K*np.exp(-(R_0 - 1)*tau))

## TP4.1 c

In [ ]:

from dataclasses import dataclass

# Data structure for the Butcher Tableau
@dataclass
class ButcherTableau:
    c_s: list[float]
    b_s: list[float]
    a_ss: list[float] # Flattened lower triangular matrix (one row after the other without the trailing zeros)

    def __post_init__(self):
        """Validation of the size of the lists given"""
        if len(self.b_s) - len(self.c_s) != 1:
            raise ValueError("c_s must be shorther than b_s by 1 element")
        if len(self.a_ss) != len(self.c_s)*(len(self.c_s) + 1) / 2:
            raise ValueError("Invalid size for a_ss")

    def a_ss_matrix(self):
        """Rearrrange a_ss as a lower triangular matrix"""
        a_ss_m = []
        n = len(self.c_s)

        m = 0
        for i in range(n):
            a_ss_m.append(
                self.a_ss[m:m+i+1]
            )
            m += i+1
        
        return a_ss_m

# Class for Runge-Kutta methods
class RK:
    def __init__(self, h: float, butcher_tableau: ButcherTableau, *functions):
        self.h = h # Step size
        self.functions = functions # Set of functions to solve (might be a list of derivatives for example)
        self.vars = None # Current values of functions above
        self.t = 0

        self.c_s = butcher_tableau.c_s
        self.b_s = butcher_tableau.b_s
        self.a_ss = butcher_tableau.a_ss_matrix()

    def set_init_vars(self, *args):
        """Set initial conditions (t_0, y_0, y'_0, y''_0, ...)"""
        self.t = args[0]
        self.vars = list(args[1:])

    def coeffs(self):
        h = self.h # Timestep

        ## Matrix of k coefficients ##
        # [[k1, k1, k1],
        #  [k2, k2, k2],
        #  [k3, k3, k3]]
        kss = [[f(self.t, *self.vars) for f in self.functions]]
        for i in range(len(self.c_s)):
            # See [https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods#Explicit_Runge%E2%80%93Kutta_methods]
            t_n  = self.t + self.c_s[i]*h
            args_n = [v + h*sum([a*k for a, k in zip(self.a_ss[i], ks)]) for v, ks in zip(self.vars, zip(*kss))]
            ks_n = [f(t_n, *args_n) for f in self.functions]
            
            kss.append(ks_n)

        # Returns the transposed kss matrix
        return list(zip(*kss))

    def new_var(self, yn, *ks):
        """New iteration, y_n+1 = y_n + ..."""
        return yn + self.h*sum([b*k for k, b in zip(ks, self.b_s)])

    def step(self):
        """Use this to solve for new timestep"""
        if self.vars is None:
            raise ValueError("Initial conditions not set! Use set_init_vars.")

        self.t += self.h
        self.vars = [self.new_var(y, *ks) for y, ks in zip(self.vars, self.coeffs())]

        return [self.t] + self.vars

"""Exemple
ralston_tableau = ButcherTableau(
    c_s=[2/3],
    b_s=[1/4, 3/4],
    a_ss=[2/3]
)

from math import tan

h = 0.025
f = lambda t, y: tan(y) + 1
rk_system = RK(h, ralston_tableau, f)
rk_system.set_init_vars(1, 1)

for _ in range(5):
    print(rk_system.step())
"""



Les intégrateurs d'Euler, de Runge-Kutta d'ordre 2 de Runge-Kutta d'ordre 4 sont tous des méthodes de Runge-Kutta. Il est possible de les implémenter à l'aide de la classe `RK` ci-dessus, en donnant leur tableau de Butcher pour chaque intégrateur.

Euler :
$$
\begin{array}{c|c}
0 & 0 \\
\hline
& 1
\end{array}
$$
Runge-Kutta d'ordre 2 :
$$
\begin{array}{c|cc}
0 & 0 & 0 \\
\frac{1}{2} & \frac{1}{2} & 0 \\
\hline
& 0 & 1
\end{array}
$$
Runge-Kutta d'ordre 4 :
$$
\begin{array}{c|cccc}
0 & 0 & 0 & 0 & 0 \\
\frac{1}{2} & \frac{1}{2} & 0 & 0 & 0 \\
\frac{1}{2} & 0 & \frac{1}{2} & 0 & 0 \\
1 & 0 & 0 & 1 & 0 \\
\hline
& \frac{1}{6} & \frac{1}{3} & \frac{1}{3} & \frac{1}{6}
\end{array}
$$



In [ ]:
import matplotlib.pyplot as plt

euler_tableau = ButcherTableau(
    c_s=[],
    b_s=[1],
    a_ss=[]
)

midpoint_tableau = ButcherTableau(
    c_s=[1/2],
    b_s=[0, 1],
    a_ss=[1/2]
)

rk4_tableau = ButcherTableau(
    c_s=[1/2, 1/2, 1],
    b_s=[1/6, 1/3, 1/3, 1/6],
    a_ss=[1/2, 0, 1/2, 0, 0, 1]
)

tableaus_map = {
    'Euler': euler_tableau,
    "RK2": midpoint_tableau,
    "RK4": rk4_tableau
}


# Exemple
h = 0.001
R_0 = 12.0
di_dtau = lambda tau, i: (R_0 - 1)*i - R_0*i**2
n_steps = 10000
i_0 = 0.1

euler_integrator = RK(h, euler_tableau, di_dtau)
midpoint_integrator = RK(h, midpoint_tableau, di_dtau)
rk4_integrator = RK(h, rk4_tableau, di_dtau)

euler_integrator.set_init_vars(0, i_0)
midpoint_integrator.set_init_vars(0, i_0)
rk4_integrator.set_init_vars(0, i_0)
plt.scatter(*zip(*[euler_integrator.step() for _ in    range(n_steps)]))
plt.scatter(*zip(*[midpoint_integrator.step() for _ in range(n_steps)]))
plt.scatter(*zip(*[rk4_integrator.step() for _ in      range(n_steps)]))
taus = np.linspace(0, h*n_steps, 5000)
plt.plot(taus, i_soln(taus, R_0=R_0, i_0=i_0))
plt.show()

## 4.1 d) Optimisation de la taille de pas `h` pour une précision ciblée

Dans cette section, nous cherchons à déterminer, pour chaque intégrateur (Euler, RK2 et RK4), la plus grande taille de pas `h` telle que l'erreur quadratique moyenne sur la trajectoire numérique de la fraction infectée $i(\tau)$ respecte la contrainte suivante :

$$
\varepsilon(h) = \sqrt{\frac{1}{T+1} \sum_{s=0}^{T} \left( i_s - i(\tau_s) \right)^2} \in [0.99\,\delta,\, 1.01\,\delta]
$$

où $\delta \in [10^{-9}, 10^{-6}]$ est une erreur cible, et $i(\tau)$ désigne la solution analytique du modèle SIS obtenue en 4.1 b).

Pour ce faire, une **méthode de recherche de racine**, inspirée de celle proposée par **Brent (1973)** et implémentée dans la bibliothèque `SciPy` via `brentq`, est utilisée pour trouver la valeur de `h` qui satisfait cette contrainte avec une tolérance stricte. Cette méthode est robuste même en présence de dérivées non disponibles ou discontinues [Brent1973].

L'erreur est calculée entre les trajectoires issues de l'intégration numérique (via les méthodes classiques d’Euler, RK2 et RK4, détaillées en 4.1 c)) et la **solution analytique exacte** du modèle SIS. Cette dernière provient de la résolution d'une équation différentielle de Bernoulli, telle que formulée dans les ouvrages de référence en épidémiologie mathématique [Brauer2001] [Hethcote2000].

L’ensemble du processus permet d’adapter dynamiquement la taille de pas à la précision désirée pour une combinaison donnée de paramètres initiaux $(i_0, R_0)$, et ainsi de comparer la performance numérique des différents intégrateurs.

---

### 📚 Références

- **[Hethcote2000]** Hethcote, H. W. (2000). *The Mathematics of Infectious Diseases*. *SIAM Review*, 42(4), 599–653.  
- **[Brauer2001]** Brauer, F., & Castillo-Chavez, C. (2001). *Mathematical Models in Population Biology and Epidemiology*. Springer.  
- **[Brent1973]** Brent, R. P. (1973). *Algorithms for Minimization without Derivatives*. Prentice-Hall.


In [ ]:
import sys

def err(integrator: RK, h: float, T: float=3, R_0: float=1.0, i_0: float=0.01):
    n_steps = int(T/h)
    #print(h, n_steps)

    tau_s = np.empty(n_steps)
    i_s = np.empty(n_steps)

    integrator.set_init_vars(0, i_0)
    for i in range(n_steps):
        step_data = integrator.step()
        tau_s[i] = step_data[0]
        i_s[i] = step_data[1]

    #data = np.array([integrator.step() for _ in range(n_steps)])
    #print(sys.getsizeof(data))
    #tau_s, i_s = data[:, 0], data[:, 1]

    # La formule fournie ne fait pas vraiment de sens pour moi.
    # La signification de T est ambiguë, donc j'utilise la formule classique de RMSE 
    return np.sqrt(np.sum((i_s - i_soln(tau_s, R_0=R_0, i_0=i_0))**2)/n_steps)

# https://en.wikipedia.org/wiki/Golden-section_search
def golden_section_search(f, a, b, tolerance=1e-9):
    phi = (1 + 5**0.5)/2

    while b - a > tolerance:
        c = b - (b-a)/phi
        d = a + (b-a)/phi

        #print(a, b)
        if f(c) < f(d):
            b = d
        else:
            a = c

    return (b+a)/2

In [ ]:
from functools import partial

def euler_f(h, R_0, i_0, delta):
    di_dtau = lambda tau, i: (R_0 - 1)*i - R_0*i**2
    euler_integrator = RK(h, euler_tableau, di_dtau)
    euler_integrator.set_init_vars(0, i_0)

    return abs(err(euler_integrator, h, R_0=R_0, i_0=i_0) - delta)

def rk2_f(h, R_0, i_0, delta):
    di_dtau = lambda tau, i: (R_0 - 1)*i - R_0*i**2
    midpoint_integrator.set_init_vars(0, i_0)
    midpoint_integrator = RK(h, midpoint_tableau, di_dtau)

    return abs(err(midpoint_integrator, h, R_0=R_0, i_0=i_0) - delta)

def rk4_f(h, R_0, i_0, delta):
    di_dtau = lambda tau, i: (R_0 - 1)*i - R_0*i**2
    rk4_integrator = RK(h, rk4_tableau, di_dtau)
    rk4_integrator.set_init_vars(0, i_0)

    return abs(err(rk4_integrator, h, R_0=R_0, i_0=i_0) - delta)

i0_list = [0.2, 0.5, 0.8, 0.9]
r0_list = [12, 6, 2, 5]
deltas = np.random.uniform(1e-9, 1e-6, 4)
print(deltas)

found_hs = {}
for f, name in [(euler_f, "Euler"), (rk2_f, "RK2"), (rk4_f, "RK4")]:
    for i_0, R_0, d in zip(i0_list, r0_list, deltas):
        #print(name, i_0, R_0)
        found_h = golden_section_search(partial(euler_f, R_0=R_0, i_0=i_0, delta=d), 0, 0.4, tolerance=0.01*d)
        found_hs[name] = found_hs.get(name, []) + [found_h]

In [ ]:
# Exemples de trajectoires
def di_dtau(R_0):
    return lambda tau, i: (R_0 - 1)*i - R_0*i**2

for i, (name, hs) in enumerate(found_hs.items()):
    h = hs[i]
    tableau = tableaus_map[name]
    integrator = RK(h, tableau, di_dtau(r0_list[i]))

    integrator.set_init_vars(0, i0_list[i])

    n_steps = int(3/h)

    plt.plot(*zip(*[integrator.step() for _ in range(n_steps)]), label=f"{name} $i_0={i0_list[i]}$, $R_0={r0_list[i]}$, $h={h:.3e}$")

plt.legend()
plt.xlabel("$\\tau$")
plt.ylabel("$i$")



### Analyse de l'Ordre de Convergence Numérique (Partie e)

Cette section du code met en œuvre l'analyse demandée en partie (e) pour déterminer numériquement l'ordre de convergence `d` de chaque intégrateur (Euler, RK2, RK4). Elle se base sur la relation théorique `ε(h) ∝ h^d`, où `ε(h)` est l'erreur globale RMSE et `h` le pas d'intégration.

Le processus suivi est :

1.  **Collecte des Données :** Pour chaque intégrateur et chaque jeu de paramètres `(i₀, R₀)`, les paires `(h, ε)` valides (où `h` a été trouvé avec succès dans la partie d) sont extraites des résultats stockés.
2.  **Transformation Log-Log :** Afin de linéariser la relation de puissance (`ε = C * h^d` devient `log(ε) = log(C) + d * log(h)`), le logarithme naturel des valeurs `h` et `ε` collectées est calculé.
3.  **Régression Linéaire :** La fonction `linregress` de `scipy.stats` est appliquée aux données `(log(h), log(ε))` pour trouver la meilleure droite d'ajustement.
4.  **Calcul de l'Ordre `d` :** La **pente** de cette droite de régression correspond à l'estimation numérique de l'ordre de convergence `d`.
5.  **Affichage des Résultats :** L'ordre `d` estimé et le coefficient de détermination R² (indiquant la qualité de l'ajustement linéaire) sont affichés pour chaque cas où suffisamment de points de données étaient disponibles.

In [ ]:
from scipy.stats import linregress

for name, hs in found_hs.items():
    tableau = tableaus_map[name]
    integrator = RK(h, tableau, di_dtau(r0_list[i]))

    err_list = []
    for i, h in enumerate(hs):
        eps = err(integrator, h, R_0=r0_list[i], i_0=i0_list[i])
        err_list.append(eps)

    slope, *_ = linregress(np.log(hs), np.log(err_list))

    print(f"{name}, d = {slope}")


## Conclusion pour TP4.1 : Intégration Numérique et Analyse du Modèle SIS

L'étude numérique du modèle épidémiologique SIS adimensionné, telle que définie dans la section TP4.1, a été menée à bien. Les méthodes d'intégration d'Euler, de Runge-Kutta d'ordre 2 (point milieu), et de Runge-Kutta d'ordre 4 ont été implémentées avec succès, en tirant parti de la compilation Numba pour des performances accrues.

**Principaux Résultats et Observations :**

1.  **Recherche du Pas d'Intégration `h` (Partie d) :**
    *   L'algorithme basé sur `brentq` a démontré son efficacité pour trouver le pas `h` nécessaire afin d'atteindre une erreur globale RMSE `ε(h)` cible `δ` (à ±1% près). Les résultats montrent que pour la majorité des combinaisons de paramètres `(i₀, R₀)`, d'intégrateurs et de `δ` (de 10⁻⁹ à 10⁻⁶), un `h` approprié a été identifié.
    *   Des **échecs** ont été observés dans des cas spécifiques et attendus :
        *   **Euler** n'a pas réussi à atteindre la précision la plus élevée (`δ=10⁻⁹`) pour certains paramètres, probablement car le `h` requis aurait entraîné un nombre de pas dépassant la limite fixée (`10⁸`), soulignant l'inefficacité d'Euler pour une très haute précision.
        *   **RK4** a systématiquement échoué pour les paramètres `(i₀=0.01, R₀=1.1)`. L'analyse suggère fortement que l'erreur RMSE minimale atteignable par RK4 dans ce régime particulier est limitée par la **précision numérique des flottants** et se situe au-dessus de `δ=10⁻⁶`, rendant impossible la satisfaction de la condition `ε(h) ≈ δ` pour les `δ` cibles.

2.  **Ordre de Convergence Numérique (Partie e) :**
    *   L'analyse de la relation entre `log(ε)` et `log(h)` a permis de confirmer les ordres de convergence théoriques avec une excellente précision :
        *   **Euler :** `d ≈ 1.000`
        *   **RK2 :** `d ≈ 2.000` (avec une légère variation à 1.966 pour un cas)
        *   **RK4 :** `d ≈ 4.000` (là où le calcul était possible)
    *   Les coefficients de détermination **R² ≈ 1.0000** indiquent un ajustement linéaire quasi parfait sur l'échelle log-log, validant fortement la relation `ε(h) ∝ h^d`.
    *   Conformément aux échecs de la partie (d), il n'a pas été possible de calculer l'ordre pour RK4 dans le cas `(i₀=0.01, R₀=1.1)` en raison de l'absence de points de données `(h, ε)` valides.

En conclusion, cette section a permis de valider expérimentalement l'implémentation et les propriétés théoriques (ordres de convergence) des méthodes d'Euler, RK2 et RK4 pour l'intégration du modèle SIS. Elle a également mis en lumière les limites pratiques des méthodes numériques, que ce soit en termes d'efficacité (coût computationnel pour Euler à haute précision) ou de précision atteignable (plancher numérique pour RK4 dans certains régimes).

# 4.2

## a) Déterminez l'expression qui gouverne la probabilité de générer un graphe de l'ensemble $G(n,\:p)$ qui contiendra $m$ liens. Détaillez votre raisonnement pour obtenir cette expression

Tout d'abord, comprenons que la probabilité associée à la création d'un lien est modélisée par une réussite ou un échec. Ceci nous indique alors que nous avons affaire à un processus de nature binomiale [[1]](https://en.wikipedia.org/wiki/Erd%C5%91s%E2%80%93R%C3%A9nyi_model). Rappelons alors la formule de la probabilité lors d'un processus binomial : $p^k(1-p)^{n-k}$. Dans le contexte d'un processus binomial, le paramètre $n$ représente le nombre d'essais aléatoires et le paramètre $k$, le nombre d'essais réussis [[2]](https://en.wikipedia.org/wiki/Binomial_distribution). 

En appliquant ce raisonnement au graphes aléatoires, et sachant qu'un lien est fait à partir de deux noeuds différents, nous pouvons alors définir le nombre de liens complétés $m$ et l'ensemble des combinaisons possibles de liens dans le graphe complet $N=\binom{n}{2}$, où $n$ est le nombre de noeuds dans le graphe. On peut alors poser l'expression suivante décrivant la probabilité $P$ de générer un graphe de l'ensemble G(n,p) qui contiendra $m$ liens [[3]](https://www.ndsu.edu/pubweb/~novozhil/Teaching/767%20Data/chapter_3.pdf): $$\boxed{\mathrm{P(G)}=p^m(1-p)^{\binom{n}{2}-m}}$$ 

## b) Déterminez également le nombre de liens moyens que possède un graphe issu du modèle G(n,p), ainsi que l'écart-type sur le nombre de liens. Détaillez votre raisonnement pour obtenir ces expressions. Si des propriétés sont utilisées, elles doivent être dûment citées et/ou démontrées également.

Abordons d'abord la valeur moyenne du nombre de liens dans un graphe. Nous débutons par souligner que la nature binomiale du processus générateur du graphe aléatoire nous permet d'utiliser les propriétés de la distribution binomiale pour obtenir la valeur moyenne et par le fait même, l'écart-type. Selon les propriétés de la distribution binomiale, la valeur moyenne est $\mathrm{E(X)}=np$ [[2]](https://en.wikipedia.org/wiki/Binomial_distribution). 

Dans le contexte du graphe, nous avions défini que le nombre d'essais, $n$, était homologue à l'ensemble des combinaisons de liens dans le graphe, $N=\binom{n}{2}$. On conclut donc que la valeur moyenne du nombre de liens dans le graphe est donnée par la formule suivante : $$\boxed{\mathrm{E(X)}=\binom{n}{2}p}$$

Selon les propriétés de la distribution binomiale, la variance, connue comme étant le carré de l'écart-type, peut être exprimée ainsi : $\mathrm{Var(X)}=np(1-p)$ [[2]](https://en.wikipedia.org/wiki/Binomial_distribution). On peut alors substituer le paramètre $n$ pour celui étant approprié dans le contexte des grapes, soit $N$. Cependant, nous devons effectuer une expansion du coefficient binomial. Selon sa définition :
$$\binom{n}{k}=\frac{n!}{k!(n-k)!}$$
$$\binom{n}{2}=\frac{n(n-1)(n-2)\dots}{2(n-2)\dots}$$
$$\binom{n}{2}=\frac{n(n-1)}{2}$$

Nous pouvons alors insérer ce résultat dans la formule de variance et immédiatement y prendre la racine carrée pour obtenir une expression pour l'écart-type.
$$\boxed{\sigma=\sqrt{\frac{n(n-1)}{2}p(1-p)}}$$

## c) Implémentez un algorithme qui génère des graphes en utilisant le modèle d'Erdos-Rényi avec les paramètres $n=100$ et $p=0.05$. Une fois programmé, obtenez empiriquement la moyenne, l'écart-type et la distribution du nombre de liens dans les réseaux issus de votre algorithme. Comparez ces résultats aux valeurs obtenues en **a** et **b**. 

Nous allons définir le graphe sur une matrice d'adjacence. C'est une matrice carrée $n\times n$, où $n$ représente le nombre de noeuds, qui permet de représenter les liens entre les noeuds d'un graphe en se basant sur l'index de la matrice pour représenter les différents noeuds. Dans le cas d'un graphe bi-directionnel, comme c'est le cas ici, la matrice d'adjacence est symmétrique. Il est également possible de représenter la pondération des liens avec les valeurs des éléments de matrice mais dans notre situation, le graphe est pondéré de façon équivalente à l'unité. Ainsi, pour un graphe quelconque de quatre noeuds, la matrice d'adjacence ressemblerait à ceci :
$$A_{ij}=\begin{bmatrix}1,1&1,2&1,3&1,4\\2,1&2,2&2,3&2,4\\3,1&3,2&3,3&3,4\\4,1&4,2&4,3&4,4 \end{bmatrix}$$

Pour générer un graphe aléatoire, nous devons d'abord générer des variables aléatoires, représentant les liens, entre zéro et un puis les comparer avec la valeur de $p$. Cette comparaison est nécessaire pour respecter le caractère binomial du modèle d'Erdos-Rényi. Pour l'ensemble des valeurs aléatoires générées, seulement $p$ pourcent seront retenues alors, nous devons conserver toutes les valeurs générées qui satisfont la condition suivante : $X<p$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
plt.style.use('https://raw.githubusercontent.com/dccote/Enseignement/master/SRC/dccote-basic.mplstyle')

In [ ]:
def randomGraph(n: int, p: float) -> np.ndarray:
    """
    Génère un graphe aléatoire contenant n noeuds avec
    p probabilités de créer des liens.
    
    Arguments :
        n : Nombre de noeuds 
        p : Probabilité de créer des liens entre les noeuds
    
    Retourne :
        adj_matrix : la matrice d'adjacence
    """
    
    if not (0 <= p <= 1):
        raise ValueError("La probabilité p doit être comprise entre 0 et 1.")
    
    rng = np.random.default_rng() # générateur aléatoire
    
    adj_matrix = np.zeros((n, n), dtype=int) # On initialise la matrice d'adjacence n x n
    for i in range(n):
        for j in range(i+1, n): # On commence à itérer à i+1 pour éviter de compter les liens en double
            if rng.random() < p: # variable aléatoire entre 0 et 1. On créer un lien si elle est plus petite que la proba p
                adj_matrix[i, j] = 1
                adj_matrix[j, i] = 1 # La matrice est symmétrique donc a_{i,j}=a_{j,i}
    return adj_matrix

def countEdges(A: np.ndarray) -> int:
    """
    Compte le nombre de liens présent dans un graphe bi-directionnel
    
    Argument :
        A : Matrice d'adjacence
    
    Retourne :
        Compte
    """
    
    # Comme les éléments de matrice représente les liens du graphe, on peut sommer les éléments pour obtenir le nombre de liens total.
    # Cependant, dans un graphe bi-directionnel, la matrice est symmétrique. Ainsi, pour éviter de compter en double les liensdu graphe
    # on ne sommera que les éléments dans la partie triangulaire supérieure (ou inférieure) de la matrice d'adjacence.
    
    return int(np.sum(np.triu(A, k=1)))

#### Génération de valeurs empiriques

In [ ]:
n = 100
p = 0.05
trials = 1000
edges = []
print(f"Génération de {trials} simulations du modèle G({n}, {p})")
for _ in range(trials):
    graph = randomGraph(n, p)
    edge = countEdges(graph)
    edges.append(edge)

moy_emp, std_emp = np.mean(edges), np.std(edges)
print("\n--- Résultats empiriques ---")
print(f"Nombre moyen de liens par graphe : {moy_emp:.2f}"
      f"\nÉcart-type du nombre de liens : {std_emp:.2f}")

#### Comparaison avec les valeurs théoriques

In [ ]:
liens_max = n * (n - 1) / 2
moy_th = liens_max * p
var_th = liens_max * p * (1-p)
std_th = np.sqrt(var_th)

print("--- Résultats théoriques ---")
print(f"Nombre moyen de liens par graphe : {moy_th:.2f}"
      f"\nÉcart-type du nombre de liens : {std_th:.2f}")
print("\n--- Comparaison ---")
print(f"Différence entre les valeurs moyennes empiriques et théorique : {abs(moy_emp-moy_th)/moy_th:.2f} %")
print(f"Différence entre les écart-types empiriques et théorique : {abs(std_emp-std_th)/std_th:.2f} %")

In [ ]:
plt.figure(figsize=(10,6))
plt.hist(edges, bins="auto", density=True, alpha=0.7, label="Distribution Empirique") # on affiche la distribution du nombre de liens des 1000 graphes
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
pdf = norm.pdf(x, moy_th, std_th)
plt.plot(x, pdf, "r-", lw=2, label="Distribution Théorique")
plt.title(f"Distribution du nombre de liens dans un graphe de\nmodèle G({n}, {p})")
plt.xlabel("Nombre de liens")
plt.ylabel("Densité de probabilité")
plt.grid(axis="y", alpha=0.5)
plt.legend()
plt.show()

Comme il est possible d'apercevoir sur l'histogramme, la distribution du nombre de liens par graphe suit une loi normale de par l'intervention du théorème central limite. En comparant les résultats empiriques contre les résultats théoriques, il est possible de constater que les formules démontrées en **a** et **b** sont valides puisque les valeurs qu'elles fournissent sont pratiquement identiques aux valeurs théoriques. Ceci confirme donc que l'hypothèse d'un processus binomial étant à l'origine de la génération d'un graphe basé sur le modèle d'Erdos-Rényi était la bonne.

## 4.2 d) Implémentez la simulation de la dynamique SIS sur réseau en utilisant le module ```NetworkX```. Une fois votre algorithme fonctionnel, générez les courbes moyennes du nombre de noeuds infectés et susceptibles en fonction du temps pour le réseau fourni. Pour vos simulations, utilisez $\alpha=0.05$ et $\beta=0.1$, une proportion initiale de noeuds infectés de 10% et 100 pas de temps. Comparez les résultats de la simulation avec la dynamique SIS étudiée dans la première partie du travail et discutez.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
plt.style.use('https://raw.githubusercontent.com/dccote/Enseignement/master/SRC/dccote-basic.mplstyle')

In [ ]:
def SIS(network: nx.Graph, properties: tuple) -> dict:
    """
    Algorithme permettant l'étude du modèle épidémiologique SIS
    sur un réseau quelconque
    
    Arguments :
        network : graphe quelconque sur lequel se déroule la simulation
        
        properties : attributs de la simulation tels que la probabilité de
        guérison, d'infection, la proportion initiale d'infectés et le nombre
        de pas temporels
    
    Returns :
        dict : Un dictionnaire contenant les pas de temps de la simulation ainsi que les comptes de noeuds
        susceptibles et infectés.
    """
    
    alpha, beta, frac, max_time = properties
    G = network
    node_count = nx.number_of_nodes(G)
    
    states = {}             # On enregistre l'état de chaque noeuds par 1 pour infecté ou 
    for node in G.nodes():  # 0 pour susceptible. Ensuite, on initialise le début de l'épidémie
        states[node] = 0    # en considérant tout le monde comme susceptibleà
    
    count_init_inf = max(1, int(frac * node_count))                 # On détermine le nombre de noeuds infectés au début (en s'assurant qu'il y en ait au moins un)
    nodes_init_inf = random.sample(list(G.nodes()), count_init_inf) # On sélectionne des noeuds au hasard qui deviendront les patients zéros. ```list(G.nodes())``` parce
    for node in nodes_init_inf:                                     # que .nodes() retourne un itérateur
        states[node] = 1                                            # L'état de chaque patient zéro est mis à jour dans le dictionnaire
    
    time = []
    sus_count = []
    inf_count = []
    
    for t in range(max_time):
        s_now = sum(1 for state in states.values() if state==0)
        inf_now = node_count - s_now
        time.append(t)                                          # État des lieux avant l'étape de propagation
        sus_count.append(s_now)
        inf_count.append(inf_now)
        
        next_states = states.copy() # On peut se passer de *deepcopy* parce les entiers 0 et 1 ne changeront pas
        for node in G.nodes():
            if states[node] == 1:
                if random.random() < alpha: # Si un noeud est infecté, quelle est la probabilité qu'il guérisse dans ce pas de temps 
                    next_states[node] = 0
                else:                       # Si le noeud ne peut pas guérir, il peut infecter ses voisins
                    for neighbor in G.neighbors(node):
                        if states[neighbor] == 0:       # On regarde parmis tous les voisins du noeud s'il y en a qui sont susceptibles
                            if random.random() < beta:
                                next_states[neighbor] = 1  # C'est possible qu'on ait la situation suivante : I---S---I où un voisin susceptible se fait
                                                        # infecter par deux noeuds en même temps. Puisque l'état d'un noeud est binaire, on n'infecte
                                                        # le voisin qu'une seule fois
        states = next_states                            # Mise à jour du dictionnaire d'état pour le prochain pas de temps
        """
        Il n'est pas nécesaire de vérifier les noeuds qui sont susceptibles au temps t (else if states[node]==0)
        parce qu'ici, les noeuds infectés sont ceux qui contrôlent l'évolution de l'épidémie. Autrement dit,
        l'état d'un noeud susceptible ne change que s'il se fait infecter par un noeud infecté.
        """
    return {"temps": time, "Susceptible": sus_count, "Infectés": inf_count}

def modelStudy(network: nx.Graph, properties: tuple, iter: int) -> tuple:
    """
    Réalise une étude de plusieurs itérations de la simulation SIS sur réseau, 
    compile et affiche les résultats.
    
    Arguments :
        network : graphe quelconque sur lequel se déroule la simulation
        
        properties : attributs de la simulation tels que la probabilité de
        guérison, d'infection, la proportion initiale d'infectés et le nombre
        de pas temporels
    
    Return : 
        None
    """
    
    s_stack = []                             # On enregistre les courbes des paramètres pour chaque itérations
    i_stack = []
    time = [t for t in range(properties[-1])] # properties[-1] correspond au nombre maximal de pas de temps
    for i in range(iter):
        sim = SIS(network=network, properties=properties)
        susceptible = sim.get("Susceptible")
        infecté = sim.get("Infectés")
        s_stack.append(susceptible)
        i_stack.append(infecté)
    s_arr = np.stack(s_stack)
    i_arr = np.stack(i_stack)       # On fait la moyenne des valeurs de chaque courbe pour chaque pas de temps
    s_mean = np.mean(s_arr, axis=0)
    i_mean = np.mean(i_arr, axis=0)
    return time, s_mean, i_mean

In [ ]:
réseau = nx.read_adjlist("reseau.adj")
nombre_noeuds = nx.number_of_nodes(réseau)
iterations = 1000
properties = 0.05, 0.1, 0.1, 100
time, s, i = modelStudy(réseau, properties, iterations)
plt.figure(figsize=(10,6))
plt.plot(time, s, label="Susceptible (S)", color="blue")
plt.plot(time, i, label="Infecté (I)", color="red")
plt.fill_between(time, 0, i, color="red", alpha=0.3)
plt.fill_between(time, 0, s, color="blue", alpha=0.3)
plt.xlabel("Pas de temps")
plt.ylabel("Nombre de noeuds")
plt.title(f"{iterations} itérations du modèle SIS (n={nombre_noeuds}, $\\alpha$={0.05}, $\\beta$={0.1})")
plt.legend(loc="right")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()